In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../data/Raw/Louisville_Metro_KY_-_Library_Collection_Inventory_.csv')

In [ ]:
df.head()

## Cleaning the Data  
The dataset contained several issues that must be addressed before analysis:  
- The 'BibNum', 'Author', 'ISBN', 'PublicationDate', and 'ReportDate' (if it is one date throughout the column) columns needed to be dropped as they were not necessary to answer questions I have about the data.
- Renamed leftover columns to match syntax used in other notebooks and sqlite.

In [ ]:
df.info()

In [ ]:
df['ReportDate'].unique()

In [ ]:
df.drop(['ReportDate'], axis=1, inplace=True)

In [ ]:
df.drop(['BibNum','Author', 'ISBN', 'PublicationYear'], axis=1, inplace=True)

In [ ]:
df = df.rename(columns={'ObjectId': 'item_id', 'Title':'title', 'ItemType':'item_type', 'ItemCollection': 'item_collection', 'ItemLocation':'item_location', 'ItemPrice':'item_price'})

In [ ]:
df.info()

## EDA
Missing data was handled as shown below:
- The row with 'Title' == NaN was dropped as it contains very little data.
- Due to a very low percentage of 'ItemCollection' == NaN and the wide variety of options regarding 'ItemCollection', it was left as is.
- For instances where 'ItemPrice' is equal to 0, it was left as is since there is no way to verify from the data if the data is missing or is equal to 0 as not all items in the library collection have an item price (donations).

Any visualizations using columns with missing data will be noted in the visualization markdown.

Regarding duplicated rows, rows cannot be validated as duplicated data or as simply duplicate items in the library inventory.

In [ ]:
df.isna()

df.isna().sum()

In [ ]:
df.loc[df['title'].isna()]

In [ ]:
df = df.drop(243965)

In [ ]:
df.isna().sum()

In [ ]:
df.loc[df['item_collection'].isna()]

In [ ]:
df.item_collection.unique()

In [ ]:
item_collection_na = df.item_collection.isna().sum()
total_rows = df.shape[0]
per_item_collection_na = (item_collection_na / total_rows) * 100
per_item_collection_na

In [ ]:
zero_item_price = df[df['item_price'] == 0]
zero_item_price

In [ ]:
df['item_price'] = df['item_price'].apply(lambda x: f'${x:,.2f}')

In [ ]:
df[df['item_price'] == 0.0]

In [ ]:
duplicated_mask = df.duplicated(keep=False)
duplicated_rows = df[duplicated_mask]
duplicated_rows

## EDA Plots (Item Totals)
Total Items by Library Branch:
- Bar charts were used in this section to show the distribution of the library collection as a whole and the book collection across all the library branches. A simple blue color was chosen for the bar charts used for the visualization folder. The goal was to keep the charts readable and focused on their objective.
- Some 'ItemLocation' entries needed to be combined to get a total for the branch (i.e. 'Remote Shelving-Shawnee' to 'Shawnee').
- The 'ItemLocation' called 'Content Management' was replaced with the standard location, 'Main,' as it only applied to 3 items.
- A plot without the Main branch was included as it has a disproportionate amount of items compared to the other branches. I would like to see a clearer comparison of those branches.
- A scattered bar chart was not very helpful due to the large amount of item types, but it did help confirm that majority of the item types that are not books are located at the Main branch.   
- A bar chart showing only book items displayed an exact trend as seen in the bar chart of all item types in which the Main branch has the vast majority of items and the branches went in the same order with book count.

In [ ]:
total_item_count = df['item_location'].value_counts()
total_item_count

plt.bar(total_item_count.index, total_item_count.values)
plt.xticks(rotation=45, ha='right')

plt.xlabel('Library Branch')
plt.ylabel('Total Item Count')
plt.title('Total Amount of Items at Each Library Branch')

plt.show()

In [ ]:
df['item_location'] = df['item_location'].replace('Remote Shelving - Shawnee', 'Shawnee')
df['item_location'] = df['item_location'].replace('Childrens Bookmobile', 'Bookmobile')
df['item_location'] = df['item_location'].replace('Adult Bookmobile', 'Bookmobile')
df['item_location'] = df['item_location'].replace(['Remote Shelving - Main', 'Childrens Main Library', 'Main Teen'], 'Main')

In [ ]:
df[df['item_location'] == 'Content Management'].value_counts()

In [ ]:
df['item_location'] = df['item_location'].replace('Content Management', 'Main')

In [ ]:
total_item_count = df['item_location'].value_counts()
total_item_count

plt.bar(total_item_count.index, total_item_count.values, color="#1A6D96")
plt.xticks(rotation=45, ha='right')

plt.xlabel('Library Branch')
plt.ylabel('Total Item Count')
plt.title('Total Amount of Items at Each Library Branch')

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../plots/TotalItemAtEachBranchBar.png')
plt.show()

In [ ]:
items_per_branch = df.groupby('item_location')['item_type'].value_counts().reset_index().sort_values(by='count', ascending=True)
items_per_branch

In [ ]:
pivot_df = items_per_branch.pivot(
    index='item_location',
    columns='item_type',
    values='count'
).fillna(0)

In [ ]:
ax = pivot_df.plot(kind='bar', stacked=True, figsize=(10,6))

ax.set_title('Item Types By LFPL Branch')

ax.legend(
    title='Item Type',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()

In [ ]:
item_count_non_Main = (
    df[df['item_location'] != 'Main']['item_location']
    .value_counts())


plt.bar(item_count_non_Main.index, item_count_non_Main.values, color="#1A6D96")
plt.xticks(rotation=45, ha='right')

plt.xlabel('Library Branch')
plt.ylabel('Total Item Count')
plt.title('Total Amount of Items at Each LFPL Branch (Excluding Main)')

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../plots/TotalItemAtEachBranchExceptMainBar.png')
plt.show()

In [ ]:
df.item_type.unique()

In [ ]:
book_itemtype = df[df['item_type'] == 'Book']
book_count = book_itemtype.groupby('item_location')['item_type'].size().sort_values(ascending=False)
book_count

plt.bar(book_count.index, book_count.values, color="#1A6D96")
plt.xticks(rotation=45, ha='right')

plt.xlabel('Library Branch')
plt.ylabel('Total Book Count')
plt.title('Total Amount of Books at Each Library Branch')

ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../plots/TotalBooksAtEachBranchBar.png')
plt.show()

## EDA Plots (Geographic Coverage)
Are there areas in Louisville that are underserved in regard to the library?
- A Pin Map was a helpful visual when looking at how the library branches impact the geography of the Louisville area. A light grey background was chosen in order to highlight the smaller libraries with the lighter orange pins. The orange cmap selection allowed the pins to be more visible.
- The Main branch has the largest collection items, which makes sense as it is the oldest branch and also holds ownership over many of the electronic items.
- The Bookmobile location was not included as it does not have a set geographic location.

In [ ]:
import geopandas as gpd
import matplotlib.patheffects as pe


In [ ]:
lfpl_loc = gpd.read_file("../data/Raw/Louisville_KY_Free_Public_Libraries/Louisville_KY_Free_Public_Libraries.shp")
inventory = pd.read_csv('../data/Clean/clean_lfpl_inventory.csv')
zipcodes = gpd.read_file("../data/Raw/Jefferson_County_KY_ZIP_Codes/Jefferson_County_KY_ZIP_Codes.shp")

In [ ]:
book_itemtype = inventory[inventory['item_type'] == 'Book']
book_count = book_itemtype.groupby('item_location').size().reset_index(name='books')

In [ ]:
book_count = book_count[book_count["item_location"] != "Bookmobile"]
book_count


In [ ]:
import utils

In [ ]:
utils.name_fix(book_count, 'item_location', lfpl_loc, 'LFPL_NAME')

In [ ]:
book_count_map = lfpl_loc.merge(book_count, left_on="LFPL_NAME", right_on="item_location")

In [ ]:
utils.lfpl_map(zipcodes, book_count_map, 'books', "Which LFPL Branches Offer the Most Books?", pe, plt, '../plots/TotalBooksPinMap.png')

## EDA (Comparisons within the Collection)
- A simple pie chart showed a comparison of LFPL Ebook versus Physical Book holdings.
- Heatmaps were used to show how the different item collections were disbursed across the different branches. They highlight some interesting aspects of certain branches such as the teens' collection is more spread out more among the branches than the children's collection and the items labeled as 'Kentucky History' are mostly housed at the Main branch, which highlights its use an archive.

In [ ]:
df.item_type.unique()

In [ ]:
ebooks = df[df['item_type'].isin(['Ebook'])]

In [ ]:
ebook_count = ebooks.groupby('item_location')['item_type'].size().sort_values(ascending=False)
ebook_count

plt.bar(ebook_count.index, ebook_count.values)
plt.xticks(rotation=45, ha='right')

plt.xlabel('Library Branch')
plt.ylabel('Total EBook Count')
plt.title('Total Amount of EBooks at Each Library Branch')

plt.show()

In [ ]:
ebook_len = len(ebooks.value_counts())

In [ ]:
book_len = len(df[df['item_type'] == 'Book'].value_counts())

In [ ]:
plt.figure(figsize=(6,6))
plt.pie(
    [ebook_len, book_len],
    labels=None,
    startangle=90,
    colors=["#8fb8caff", "#22649aff"],
    autopct='%1.1f%%',
    textprops={'fontsize' : 12, 'weight': 'bold'})

plt.title('How Large of a Role Do Ebooks Have in Our Libraries?', fontsize=16)
plt.legend(
    labels = ['Ebooks', 'Physical books'], fontsize=12,
    loc='lower right'
)


plt.tight_layout()
plt.savefig('../plots/BooksvsEbooksPieChart.png')
plt.show()


In [ ]:
df['item_collection'].unique()

In [ ]:
childrens_words = ['Children', 'Preschool', 'Storytime']
pattern = '|'.join(childrens_words)
childrens_df = df[
    df['item_collection'].str.contains(pattern, case=False, na=False)
]


In [ ]:
childrens_df_byloc = childrens_df.groupby(['item_location', 'item_collection']).size().reset_index(name='count')
childrens_df_byloc


In [ ]:
heatmap_data = childrens_df_byloc.pivot(
    index='item_location',
    columns='item_collection',
    values='count'
).fillna(0)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, cmap='Blues')

plt.title("Children's Collection Items by Library Branch")
plt.xlabel("Collection Type")
plt.ylabel("Library Branch")

plt.tight_layout()
plt.savefig('../plots/ChildrensCollectionHeatMap.png')
plt.show()

In [ ]:
teen_words = ['Teen', 'College']
pattern = '|'.join(teen_words)
teens_df = df[
    df['item_collection'].str.contains(pattern, case=False, na=False)]

In [ ]:
teens_df_byloc = teens_df.groupby(['item_location', 'item_collection']).size().reset_index(name='count')
teens_df_byloc

In [ ]:
heatmap_data = teens_df_byloc.pivot(
    index='item_location',
    columns='item_collection',
    values='count'
).fillna(0)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, cmap='Blues')

plt.title("Teens' Collection Items by Library Branch")
plt.xlabel("Collection Type")
plt.ylabel("Library Branch")

plt.tight_layout()
plt.savefig('../plots/TeensCollectionHeatMap.png')
plt.show()

In [ ]:
adult_words = ['Adult', 'ELL', 'Mystery', 'Science', 'Western', 'International', 'Kentucky', 'Natural', 'Oversize','Holiday', 'Urban', 'Bestsellers', 'Large', 'Caldecott/Newberry', 'Government', 'Telereference', 'Big', 'Magazines']
pattern = '|'.join(adult_words)
adults_df = df[
    df['item_collection'].str.contains(pattern, case=False, na=False)]

In [ ]:
adults_df_byloc = adults_df.groupby(['item_location', 'item_collection']).size().reset_index(name='count')
adults_df_byloc

In [ ]:
heatmap_data = adults_df_byloc.pivot(
    index='item_location',
    columns='item_collection',
    values='count'
).fillna(0)

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, cmap='Blues')

plt.title("Adults' Collection Items by Library Branch")
plt.xlabel("Collection Type")
plt.ylabel("Library Branch")

plt.tight_layout()
plt.savefig('../plots/AdultsCollectionHeatMap.png')
plt.show()

In [ ]:
df.to_csv('../data/Clean/clean_lfpl_inventory.csv', index=False)